# Bias Fairness Analysis with Zero-Shot LLMs and Integrated Gradients

Zero-shot classification with Integrated Gradients explanations and fairness analysis across demographic identity groups for Responsible AI evaluation.

### Pipeline Overview
- Zero-shot toxicity/hate/offense classification using LLMs
- Integrated Gradients (IG) token attribution explanations
- Token-level attribution heatmap visualization
- Fairness metrics computation across demographic groups
- Dataset downloading utilities for social bias benchmarks

### Features
- Log-probability-based zero-shot classification (no fine-tuning required)
- Integrated Gradients for model interpretability
- Group fairness metrics (SPD, Equal Opportunity, worst-case)
- Resume-safe Parquet/CSV I/O
- GPU/CPU automatic device selection
<!-- ## File Overview
| File | Purpose |
|------|---------|
| `llm_zero_shot_explain.py` | Core zero-shot classification engine with Integrated Gradients explanations. |
| `fairness_metrics.py` | Computes per-group fairness metrics (Accuracy, F1, TPR, FPR, SPD, EOpp). |
| `download_data.py` | Download and normalize social bias datasets from Hugging Face. |
| `sample_jigsaw.py` | Convert Kaggle Jigsaw CSV to normalized Parquet format. | -->

## Download and Normalize Social-Problem Datasets

This notebook downloads, normalizes, and saves social-problem datasets (Jigsaw, Civil) into Parquet files.

Outputs are standardized with at least:
- `comment_text`
- `target`

In [ ]:
# ===== Standard library =====
import importlib.util
import itertools
import random
from collections.abc import Generator, Iterator
from pathlib import Path
from typing import Any
import os

# ===== Scientific stack =====
import numpy as np
import numpy.typing as npt
import pandas as pd

# ===== ML / DL =====
import torch
from torch.nn import functional
from transformers import AutoModelForCausalLM, AutoTokenizer

# ===== Datasets & metrics =====
from datasets import load_dataset
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

# ===== Visualization & progress =====
import matplotlib.pyplot as plt
from rich.progress import track


In [2]:
from src.download_data import (
    load_civil,
    to_parquet  
)

In [3]:
# Configuration (edit interactively)
DATASET = "jigsaw" # jigsaw or civil
STREAM = True
TAKE = 50000 # for full data = 100000, 5_000 # for jigsaw = 50000
SAMPLE = None
DATA_DIR = f"./data/{DATASET}/"
os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
OUT_PATH = f"./data/{DATASET}/{DATASET}.parquet"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

### Dataset Setup

* **CivilComments**
  Automatically downloaded via `load_dataset("google/civil_comments")` if not already cached locally. No manual action required.

* **Jigsaw Unintended Bias in Toxicity Classification**
  Must be downloaded manually from Kaggle:
  [https://www.kaggle.com/competitions/jigsaw-unintended-bias-in-toxicity-classification/data?select=train.csv](https://www.kaggle.com/competitions/jigsaw-unintended-bias-in-toxicity-classification/data?select=train.csv)

  After downloading, place `train.csv` inside:

  ```
  data/jigsaw/
  ```


In [4]:
def load_dataset(
    dataset_name,
    TAKE,
    DATA_DIR,
):
    """Load the dataset based on the dataset name and return a DataFrame."""
    if dataset_name == "civil":
        df = load_civil(stream=True, take=TAKE)
    
        if TAKE and len(df) > TAKE:
            df = df.sample(TAKE, random_state=42).reset_index(drop=True)
        return df

    if dataset_name == "jigsaw":
        df = pd.read_csv(f"{DATA_DIR}/train.csv")
        # Keep only the relevant columns
        COLS_KEEP = [
            "comment_text", "target", "severe_toxicity",
            "obscene", "identity_attack", "insult",
            "threat", "male", "female",
            "transgender", "other_gender", "heterosexual",
            "homosexual_gay_or_lesbian", "bisexual",
            "other_sexual_orientation", "christian",
            "jewish", "muslim", "hindu",
            "buddhist", "atheist", "other_religion",
            "black", "white", "asian", "latino", "other_race_or_ethnicity",
        ]

        df = df[COLS_KEEP].copy()

        if TAKE:
            df = df.sample(TAKE, random_state=42).reset_index(drop=True)

        return df

### Data Formats

### Input Datasets
Supported datasets via `download_data.py`:
- **CivilComments**: Toxicity labels with 7 toxicity sub-types

Expected columns:
```json
{
  "comment_text": "Text content to classify",
  "target": 0 or 1,  // binary toxicity label
  "severe_toxicity": 0.0-1.0,   // identity attribute scores
  "obscene": 0.0-1.0,
  "insult": 0.0-1.0,
  "thread": 0.0-1.0,
  // ... additional identity columns
}
```

In [5]:
# Load the dataset
df = load_dataset(DATASET,TAKE, DATA_DIR)
df.head()

,comment_text,target,severe_toxicity,obscene,identity_attack,insult,threat,male,female,transgender,...,muslim,hindu,buddhist,atheist,other_religion,black,white,asian,latino,other_race_or_ethnicity
0,What a breathe of fresh air to have someone wh...,0.166667,0.0,0.0,0.0,0.166667,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Your jewish friends were the ones who told you...,0.600000,0.2,0.0,0.6,0.400000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,Possible collusion by Trump and his affiliates...,0.000000,0.0,0.0,0.0,0.000000,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Exactly. We need a % of GDP spending cap at t...,0.000000,0.0,0.0,0.0,0.000000,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"By your own comment, even if some of them vote...",0.000000,0.0,0.0,0.0,0.000000,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Run it only once to save as parquet (uncomment if needed)
# OUT_DIR = Path(f"./data/{DATASET}")
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# PARQUET_PATH = OUT_DIR / f"{DATASET}.parquet"
# df.to_parquet(PARQUET_PATH, index=False)

# print(f"Saved -> {PARQUET_PATH}")

In [5]:
# to_parquet(df, OUT_PATH) # Uncomment to save as parquet

Wrote 100,000 rows -> data/civil/civil.parquet


## Zero-shot LLM Toxicity Scoring with Integrated Gradients

Performs zero-shot binary classification on social-problem text
(e.g. toxicity, hate, offense) using a causal LLM, and computes
Integrated Gradients (IG) explanations over prompt tokens.

Key features:
- Zero-shot scoring via log-probability differences
- Integrated Gradients using `inputs_embeds`
- Optional float32 stability for LayerNorm
- Parquet/CSV-safe data loading

In [4]:
from src.llm_zero_shot_explain import (
    get_device,
    load_llm,
    score_and_predict,
    integrated_gradients,
    save_heatmap,
    load_df_safely,
)

In [5]:
# Define labels for each task
LABELS = {
    "toxicity": ["toxic", "non-toxic"],
    "hate": ["hateful", "not hateful"],
    "offense": ["offensive", "not offensive"],
}

In [6]:
# ---- User configuration ----
INPUT_PATH = f"./data/{DATASET}/{DATASET}.parquet"
TEXT_COL = "comment_text"
TASK = "toxicity"
MODEL_NAME = "distilgpt2"

MAX_ROWS = 1000
IG_ROWS = 25
IG_STEPS = 32
FORCE_FLOAT32 = True
SAVE_HEATMAPS = True

OUTPUT_PATH = f"./results/{DATASET}/zs_preds.parquet"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

In [7]:
def run_scoring_with_explanations(
    df,
    text_col,
    model_name,
    task,
    input_path,
    dataset,
    max_rows=None,
    save_heatmaps=False,
    ig_rows=0,
    ig_steps=50,
    force_float32=False,
):
    """Run scoring with explanations on the given DataFrame."""
    df = df.copy()
    if max_rows is not None:
        df = df.iloc[:max_rows]

    device = get_device()
    model, tok = load_llm(model_name, device, force_float32=force_float32)

    preds = []
    ig_records = []
    score_diffs = []

    heatmap_dir = Path(f"./results/{dataset}/ig_heatmaps")
    heatmap_dir.mkdir(parents=True, exist_ok=True)

    for i in track(range(len(df)), description="Scoring"):
        text = str(df.iloc[i][text_col])[:4096]

        res = score_and_predict(model, tok, text, task)

        preds.append({
            "idx": i,
            "pred": res["pred"],
            "score": res["score"],
            "lp_pos": res["lp_pos"],
            "lp_neg": res["lp_neg"],
        })

        # ---- Run IG only for first N rows ----
        if save_heatmaps and len(ig_records) < ig_rows:
            toks, atts, prompt, ig_score = integrated_gradients(
                model, tok, text, task, steps=ig_steps
            )

            # Compare IG scalar with scoring scalar
            full_score = res["score"]
            score_diffs.append(abs(full_score - ig_score))

            img_path = heatmap_dir / f"row{i}.png"
            save_heatmap(toks, atts, str(img_path))

            ig_records.append({
                "idx": i,
                "heatmap": str(img_path),
                "prompt": prompt,
            })

    # ---- Report alignment summary ----
    if score_diffs:
        print(f"Max |full_score - ig_score|: {max(score_diffs):.6e}")

    return preds, ig_records

In [8]:
from src.llm_zero_shot_explain import format_prompt

In [ ]:
# Load DataFrame safely and run scoring with explanations
df = load_df_safely(INPUT_PATH)
preds, ig_records = run_scoring_with_explanations(
    df=df,
    text_col=TEXT_COL,
    model_name=MODEL_NAME,
    task=TASK,
    input_path=INPUT_PATH,
    dataset=DATASET,
    max_rows=MAX_ROWS,
    save_heatmaps=SAVE_HEATMAPS,
    ig_rows=IG_ROWS,
    ig_steps=IG_STEPS,
    force_float32=FORCE_FLOAT32,
)

In [ ]:
# Save predictions and IG records
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(preds).to_parquet(OUTPUT_PATH, index=False)

# Save IG records if any
if ig_records:
    pd.DataFrame(ig_records).to_parquet(
        Path(OUTPUT_PATH).with_suffix(".ig.parquet"),
        index=False,
    )

print(f"Saved predictions -> {OUTPUT_PATH}")

## Fairness Metrics Across Demographic Groups

In [12]:
from src.fairness_metrics import (
    load_df,
    binarize_labels,
    metrics_for_group,
)

In [13]:
# Load predictions
df = pd.read_parquet(f"./results/{DATASET}/zs_preds.parquet")
print(df.columns.tolist())

['idx', 'pred', 'score', 'lp_pos', 'lp_neg']


In [14]:
# ===== CONFIG =====
PREDS_PATH = f"./results/{DATASET}/zs_preds.parquet"
LABELS_FILE = f"./data/{DATASET}/{DATASET}.parquet"
LABEL_COL = "target"
POSITIVE_LABEL = 1

# Define identity columns based on dataset
if DATASET == "civil":
    IDENTITY_COLS = [
        "severe_toxicity",
        "obscene",
        "identity_attack",
        "insult",
        "threat",
    ]
elif DATASET == "jigsaw":
    IDENTITY_COLS = [
        "male", "female", "black", "white",
        "muslim", "jewish", "christian",
    ]
MIN_GROUP_SIZE = 30
OUTPUT_PATH = f"./results/{DATASET}/fairness_report.csv"

In [15]:
def prepare_fairness_dataframe(
    preds_path,
    labels_path,
    label_col,
    positive_label,
):
    """Prepare a DataFrame for fairness analysis by loading predictions and labels, merging them, and binarizing the labels."""
    
    # Load predictions
    df_preds = load_df(preds_path)

    # Ensure 'idx' column exists in predictions
    if "idx" not in df_preds.columns:
        raise ValueError("Predictions must contain 'idx' column.")

    # Load labels and merge with predictions
    if labels_path:
        df_labels = load_df(labels_path)
        if "idx" not in df_labels.columns:
            df_labels = df_labels.reset_index().rename(columns={"index": "idx"})
        df = df_preds.merge(df_labels, on="idx", how="left")
    else:
        df = df_preds.copy()

    # Check for label column
    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found.")

    # Check for prediction column
    if "pred" not in df.columns:
        raise ValueError("Missing 'pred' column in predictions.")

    # Binarize labels and predictions
    y_true = binarize_labels(df[label_col], positive_label=positive_label)
    y_pred = df["pred"].astype(int).values

    return df, y_true, y_pred

In [16]:
def evaluate_single_identity(
    df,
    y_true,
    y_pred,
    identity,
    min_group_size,
):
    """Evaluate fairness metrics for a single identity column by calculating metrics for groups defined by the identity and computing disparities between them."""
    
    # Check if identity column exists
    if identity not in df.columns:
        print(f"[warn] identity column '{identity}' missing; skipping")
        return [], None

    # Create membership array for the identity group (1 if member, 0 if not)
    membership = (
        pd.to_numeric(df[identity], errors="coerce")
        .fillna(0)
        .gt(0)
        .astype(int)
        .values
    )

    group_rows = []

    # Evaluate metrics for both groups (members and non-members)
    for val in (0, 1):
        mask = membership == val
        n = int(mask.sum())

        if n < min_group_size:
            group_rows.append({
                "identity": identity,
                "group": f"{identity}={val}",
                "n": n,
                "skipped": True,
            })
            continue

        metrics = metrics_for_group(y_true[mask], y_pred[mask])

        group_rows.append({
            "identity": identity,
            "group": f"{identity}={val}",
            "n": n,
            "skipped": False,
            **metrics,
        })

    # Calculate disparities between the two groups if both are valid
    g0 = next((r for r in group_rows if r["group"].endswith("=0") and not r["skipped"]), None)
    g1 = next((r for r in group_rows if r["group"].endswith("=1") and not r["skipped"]), None)

    # Calculate disparities if both groups are valid
    disparity = None
    if g0 and g1:
        disparity = {
            "identity": identity,
            "SPD": g1["pos_rate"] - g0["pos_rate"],
            "EOpp_diff": g1["tpr"] - g0["tpr"],
            "n_A0": g0["n"],
            "n_A1": g1["n"],
        }

    return group_rows, disparity


In [17]:
def evaluate_fairness_by_identity(
    preds_path,
    labels_path,
    label_col,
    positive_label,
    identity_cols,
    min_group_size=50,
):
    """Evaluate fairness metrics by identity by preparing the DataFrame, evaluating each identity column, and calculating disparities between groups."""

    # Prepare the DataFrame and extract true labels and predictions
    df, y_true, y_pred = prepare_fairness_dataframe(
        preds_path,
        labels_path,
        label_col,
        positive_label,
    )

    rows = []
    per_identity = []

    # Evaluate each identity column and calculate disparities
    for identity in identity_cols:
        group_rows, disparity = evaluate_single_identity(
            df,
            y_true,
            y_pred,
            identity,
            min_group_size,
        )

        rows.extend(group_rows)

        if disparity:
            per_identity.append(disparity)

    return pd.DataFrame(rows), pd.DataFrame(per_identity)


In [18]:
# Run fairness evaluation
group_df, disparity_df = evaluate_fairness_by_identity(
    preds_path=PREDS_PATH,
    labels_path=LABELS_FILE,
    label_col=LABEL_COL,
    positive_label=POSITIVE_LABEL,
    identity_cols=IDENTITY_COLS,
    min_group_size=MIN_GROUP_SIZE,
)

In [19]:
def write_fairness_reports(
    rows,
    per_identity,
    output_path,
):
    """Write fairness reports to CSV files."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Per-group metrics
    rep = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)
    rep.to_csv(output_path, index=False)

    # Per-identity disparities
    has_per_identity = (
        per_identity is not None
        and (
            (isinstance(per_identity, pd.DataFrame) and not per_identity.empty)
            or (isinstance(per_identity, list) and len(per_identity) > 0)
        )
    )

    pi_path = output_path.with_suffix(".per_identity.csv")

    if has_per_identity:
        pi = (
            per_identity
            if isinstance(per_identity, pd.DataFrame)
            else pd.DataFrame(per_identity)
        )
        pi.to_csv(pi_path, index=False)

        worst_spd = (
            pi["SPD"].abs().max()
            if "SPD" in pi.columns and not pi.empty
            else np.nan
        )
        worst_eopp = (
            pi["EOpp_diff"].abs().max()
            if "EOpp_diff" in pi.columns and not pi.empty
            else np.nan
        )
    else:
        pi = None
        worst_spd = np.nan
        worst_eopp = np.nan

    # Safe non-skipped filtering
    if not rep.empty and "skipped" in rep.columns:
        non_skipped = rep[~rep["skipped"]]
    else:
        non_skipped = rep

    worst_acc = (
        non_skipped["acc"].min()
        if "acc" in non_skipped.columns and not non_skipped.empty
        else np.nan
    )

    worst_f1 = (
        non_skipped["f1"].min()
        if "f1" in non_skipped.columns and not non_skipped.empty
        else np.nan
    )

    # Summary
    summary = pd.DataFrame(
        [
            {
                "WorstAbsSPD": worst_spd,
                "WorstAbsEOpp": worst_eopp,
                "WorstGroupAcc": worst_acc,
                "WorstGroupF1": worst_f1,
            }
        ]
    )

    summary_path = output_path.with_suffix(".summary.csv")
    summary.to_csv(summary_path, index=False)

    # Logging
    print("Saved:")
    print(" - Per-group metrics:", output_path)
    print(" - Per-identity disparities:", pi_path)
    print(" - Summary:", summary_path)

In [20]:
# Write fairness reports
write_fairness_reports(
    rows=group_df,
    per_identity=disparity_df,
    output_path=OUTPUT_PATH,
)

Saved:
 - Per-group metrics: results/jigsaw/fairness_report.csv
 - Per-identity disparities: results/jigsaw/fairness_report.per_identity.csv
 - Summary: results/jigsaw/fairness_report.summary.csv


## Stage 4 — Visualize Fairness Results

In this stage, we analyze whether the **zero-shot LLM toxicity classifier** exhibits systematic disparities across different **toxicity subtypes** present in the CivilComments dataset.

Rather than demographic identities (e.g., gender or religion), CivilComments provides **content-based subgroup labels** such as *insult*, *threat*, and *identity attack*. These can be treated as binary group indicators to evaluate fairness across different forms of toxic content.

### ⚖️ Fairness Metrics Used

#### **Statistical Parity Difference (SPD)**

$$
\text{SPD} = P(\hat{Y}=1 \mid A=1) - P(\hat{Y}=1 \mid A=0)
$$

**Interpretation**

* Measures whether the model predicts *toxicity* more often for comments **mentioning an identity**.
* SPD = 0 → parity
* SPD > 0 → identity-associated comments are flagged *more toxic*
* SPD < 0 → identity-associated comments are flagged *less toxic*

---

### **Equal Opportunity Difference (EOpp)**

$$
\text{EOpp} = \text{TPR}_{A=1} - \text{TPR}_{A=0}
$$

**Interpretation**

* Measures whether the model correctly detects *true toxic comments* at the same rate across groups.
* Sensitive to **false negatives** for protected identities.
* Non-zero values indicate uneven error rates.

In [21]:
# Define output paths
OUTPUT_DIR = Path(f"results/{DATASET}")
SUMMARY_PATH = OUTPUT_DIR / "fairness_report.summary.csv"
PER_IDENTITY_PATH = OUTPUT_DIR / "fairness_report.per_identity.csv"

The table below summarizes the **worst observed disparities** across all toxicity subtypes:

* **WorstAbsSPD**: largest absolute statistical parity difference
* **WorstAbsEOpp**: largest absolute equal opportunity difference
* **WorstGroupAcc / WorstGroupF1**: lowest performance among all groups

These values help identify whether *any* subgroup experiences disproportionately poor or biased treatment.

In [ ]:
# Load and display summary report
summary_df = pd.read_csv(SUMMARY_PATH)
summary_df

### 🧩 Per-Identity (Subtype) Fairness Breakdown

For each toxicity subtype (e.g., `identity_attack`, `insult`, `threat`), we compute:

* **SPD**: difference in predicted toxicity rate
* **EOpp_diff**: difference in true positive rates
* **n_A0 / n_A1**: group sizes (absence vs presence of subtype)

This breakdown allows us to localize where disparities arise instead of relying only on aggregate statistics.


In [ ]:
# Load and display per-identity report
pi = pd.read_csv(PER_IDENTITY_PATH)
pi

### 📈 Visualization: Statistical Parity Difference (SPD)

The bar chart below visualizes **SPD per toxicity subtype**.

* Each bar corresponds to one subtype (e.g., *identity_attack*).
* The horizontal dashed line at 0 indicates **perfect statistical parity**.
* Positive SPD values mean the model predicts *toxicity* more often when that subtype is present.

This visualization makes it easy to identify which forms of toxic content are most strongly associated with biased predictions.

In [ ]:
# Plot Statistical Parity Difference (SPD) by identity
plt.figure(figsize=(8, 4))
pi.sort_values("SPD").plot(
    x="identity",
    y="SPD",
    kind="bar",
    legend=False,
)
plt.axhline(0, linestyle="--", linewidth=1)
plt.ylabel("Statistical Parity Difference (SPD)")
plt.title("Fairness: Statistical Parity Difference by Identity")
plt.tight_layout()

out_path = OUTPUT_DIR / f"{DATASET}_SPD.png"
plt.savefig(out_path, dpi=200)
plt.show()

print(f"Saved -> {out_path}")

### 🧠 Interpretation and Takeaways (Civil dataset)

From the results:

* Certain subtypes (e.g., **identity_attack** and **severe_toxicity**) show noticeably higher SPD values.
* This suggests the zero-shot LLM is **more likely to flag toxicity** when these patterns are present, even controlling for the overall label.
* Such behavior may reflect:

  * Model sensitivity to strong lexical cues
  * Training data biases inherited by the language model
  * Over-generalization from certain toxic patterns

Importantly, this analysis does **not** imply intent or malice, but highlights where **model behavior differs systematically across content subgroups**.

### 🧠 Interpretation and Takeaways (Jigsaw dataset)

In this run:

* *female* shows the largest positive SPD
* *male* shows a smaller but non-zero disparity
* *christian* lies in between

This suggests the model is **more likely to flag comments referencing certain identities as toxic**, even without explicit ground-truth justification.

#### Key takeaways:
* The zero-shot LLM exhibits **measurable fairness disparities** on Jigsaw.
* Disparities are **identity-dependent**, not uniform.
* Even without fine-tuning, **prompt-based toxicity classification can amplify bias**.
* These results motivate:
  * Prompt calibration
  * Post-hoc thresholding
  * Explanation-based audits (e.g., Integrated Gradients)

### 🔍 Relation to Interpretability

In earlier stages, we computed **Integrated Gradients explanations** for selected examples. Combining those explanations with fairness results allows us to:

* Inspect *which tokens* drive high toxicity scores in high-SPD subgroups
* Diagnose whether explanations differ qualitatively across subtypes
* Bridge **fairness evaluation** with **model interpretability**

This closes the loop between *what the model predicts*, *how fairly it behaves*, and *why it makes those predictions*.

## Reflection & Concept Check


### Why use Integrated Gradients (IG) instead of SHAP in this notebook?

* IG operates directly on model gradients and works naturally with transformer embeddings.
* It scales better for long text sequences than SHAP.
* It avoids expensive perturbation-based sampling.
* It provides path-integrated attribution consistent with model internals.

**Think about:**
If you replaced IG with SHAP for `distilgpt2`, what would break computationally or conceptually?

### Why might a zero embedding baseline be problematic for text models?

* Zero embeddings do not correspond to any real token.
* The path from zero → actual embedding may pass through unrealistic regions of embedding space.
* Language models are not trained on zeroed embeddings.
* Attribution magnitudes may reflect embedding geometry rather than linguistic contribution.

**Follow-up:**
What alternative baselines could you try?

* Padding token embedding
* Neutral sentence baseline
* Mean embedding baseline

### What does a positive SPD actually mean in this experiment?

If SPD > 0 for `female`, does that mean:

* The dataset is biased?
* The model is biased?
* The threshold is miscalibrated?
* The subgroup distribution differs?

Explain what is *mathematically true* versus what is an *interpretation*.

### Why compute both SPD and Equal Opportunity Difference?

* SPD measures prediction rate disparity.
* EOpp measures disparity in true positive rates.
* A model can satisfy one but violate the other.

### How does interpretability complement fairness analysis?

You computed:

* IG token attributions
* Group-level fairness disparities

How could you use IG explanations to diagnose *why* a subgroup has high SPD?

## References:
Papers:
1. Integrated Gradients: Kapishnikov et al. (2021) - [Guided Integrated Gradients: An Adaptive Path Method for Removing Noise](https://openaccess.thecvf.com/content/CVPR2021/html/Kapishnikov_Guided_Integrated_Gradients_An_Adaptive_Path_Method_for_Removing_Noise_CVPR_2021_paper.html)
2. Fairness Metrics: Mehrabi et al. (2023) - [A Survey on Bias and Fairness in Machine Learning](https://ieeexplore.ieee.org/abstract/document/10771762/)
3. Zero-Shot Classification: Brown et al. (2020) - [Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165) (GPT-3 paper)

Datasets:
1. CivilComments: Borkan et al. (2019) - [Nuanced Metrics for Measuring Unintended Bias with Real Data for Text Classification](https://arxiv.org/abs/1903.04561)
2. Jigsaw Toxic Comment Classification: [Kaggle Jigsaw Unintended Bias in Toxicity Classification dataset](https://www.kaggle.com/c/jigsaw-unintended-bias-in-toxicity-classification)
3. Bias in Toxicity Detection: Sap et al. (2019) - [The Risk of Racial Bias in Hate Speech Detection](https://arxiv.org/abs/1905.12516)

